# Task 2: Interactive Choropleth — Global Installs by Category

**Requirements:**
- Interactive **Choropleth map** (Plotly) visualizing global installs by category
- Filters:
  - Show only the **top 5 app categories** (by installs)
  - **Highlight** any category where installs exceed **1 million**
  - Category name must **not start with** `A`, `C`, `G`, or `S`
- Display rule: only visible between **6 PM – 8 PM IST** (implemented in the dashboard HTML, not here)

### Important data note
The Google Play Store dataset used here has **no per-country breakdown** — it only has a single global install count per app/category. To satisfy the "global" choropleth requirement, we **estimate** a per-country split of each category's installs using each country's approximate public share of global Android users. This is clearly labeled as an **illustrative estimate**, not real per-country install data (which Google does not publish).

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pd.set_option('display.max_columns', None)

## 1. Load and clean the data

In [2]:
df = pd.read_csv('googleplaystore.csv')
df = df.drop_duplicates(subset='App', keep='first')

# Drop the known corrupt row in this dataset (shifted columns -> Category == '1.9')
df = df[~df['Category'].str.contains(r'^\d', regex=True, na=False)]

# Installs -> numeric
df['Installs_Num'] = (
    df['Installs'].astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)
df['Installs_Num'] = pd.to_numeric(df['Installs_Num'], errors='coerce')

df = df.dropna(subset=['Installs_Num', 'Category'])
print(df.shape)
df[['App','Category','Installs_Num']].head()

(9659, 14)


,App,Category,Installs_Num
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,10000
1,Coloring book moana,ART_AND_DESIGN,500000
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,5000000
3,Sketch - Draw & Paint,ART_AND_DESIGN,50000000
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,100000


## 2. Apply filters

1. Exclude any category starting with `A`, `C`, `G`, or `S`
2. Sum installs per remaining category, take the **top 5**
3. Flag categories whose total installs exceed 1,000,000 (for highlighting)

In [3]:
excluded_prefixes = ('A', 'C', 'G', 'S')

cat_totals = df.groupby('Category')['Installs_Num'].sum().sort_values(ascending=False)
cat_totals = cat_totals[~cat_totals.index.str.startswith(excluded_prefixes)]

top5 = cat_totals.head(5).reset_index()
top5.columns = ['Category', 'Total_Installs']
top5['Exceeds_1M'] = top5['Total_Installs'] > 1_000_000

top5

,Category,Total_Installs,Exceeds_1M
0,TOOLS,8001771915,True
1,PRODUCTIVITY,5793091369,True
2,PHOTOGRAPHY,4649147655,True
3,FAMILY,4427941505,True
4,VIDEO_PLAYERS,3926902720,True


## 3. Estimate a per-country install split

We distribute each category's total installs across ~25 major countries using each country's **approximate share of global Android/smartphone users** (public, widely-cited figures, not exact). This lets the choropleth show relative regional intensity while being transparent that per-country install counts are not present in the source data.

In [4]:
# Approximate share of global Android users by country (illustrative, sums to 1.0)
country_weights = {
    'IND': 0.235,  # India
    'USA': 0.095,  # United States
    'BRA': 0.075,  # Brazil
    'IDN': 0.070,  # Indonesia
    'RUS': 0.045,  # Russia
    'CHN': 0.060,  # China (Android, excl. domestic app stores)
    'MEX': 0.040,  # Mexico
    'VNM': 0.032,  # Vietnam
    'PHL': 0.030,  # Philippines
    'PAK': 0.028,  # Pakistan
    'NGA': 0.026,  # Nigeria
    'BGD': 0.024,  # Bangladesh
    'TUR': 0.022,  # Turkey
    'DEU': 0.020,  # Germany
    'GBR': 0.018,  # United Kingdom
    'FRA': 0.017,  # France
    'THA': 0.016,  # Thailand
    'EGY': 0.015,  # Egypt
    'ITA': 0.014,  # Italy
    'ESP': 0.013,  # Spain
    'ZAF': 0.012,  # South Africa
    'KOR': 0.011,  # South Korea
    'JPN': 0.010,  # Japan
    'ARG': 0.010,  # Argentina
    'CAN': 0.009,  # Canada
    'AUS': 0.008,  # Australia
    'POL': 0.008,  # Poland
    'UKR': 0.007,  # Ukraine
    'MYS': 0.007,  # Malaysia
    'SAU': 0.007,  # Saudi Arabia
}

total_weight = sum(country_weights.values())
country_weights = {k: v / total_weight for k, v in country_weights.items()}  # normalize to 1.0

print(f"Countries: {len(country_weights)}, weights sum to {sum(country_weights.values()):.3f}")

Countries: 30, weights sum to 1.000


In [5]:
rows = []
for _, r in top5.iterrows():
    for iso3, weight in country_weights.items():
        rows.append({
            'Category': r['Category'],
            'ISO3': iso3,
            'Estimated_Installs': r['Total_Installs'] * weight,
            'Exceeds_1M': r['Exceeds_1M']
        })

geo_df = pd.DataFrame(rows)
geo_df.head(10)

,Category,ISO3,Estimated_Installs,Exceeds_1M
0,TOOLS,IND,1.910992e+09,True
1,TOOLS,USA,7.725288e+08,True
2,TOOLS,BRA,6.098912e+08,True
3,TOOLS,IDN,5.692317e+08,True
4,TOOLS,RUS,3.659347e+08,True
5,TOOLS,CHN,4.879129e+08,True
6,TOOLS,MEX,3.252753e+08,True
7,TOOLS,VNM,2.602202e+08,True
8,TOOLS,PHL,2.439565e+08,True
9,TOOLS,PAK,2.276927e+08,True


## 4. Build the interactive Choropleth (Plotly)

A dropdown lets the viewer switch between the 5 categories. Categories that exceed 1M installs (all 5, here) are marked with a ★ in their dropdown label and use a warmer color scale as a highlight cue.

In [6]:
categories = top5['Category'].tolist()

fig = go.Figure()

for i, cat in enumerate(categories):
    sub = geo_df[geo_df['Category'] == cat]
    exceeds = sub['Exceeds_1M'].iloc[0]
    colorscale = 'Oranges' if exceeds else 'Blues'
    fig.add_trace(go.Choropleth(
        locations=sub['ISO3'],
        z=sub['Estimated_Installs'],
        locationmode='ISO-3',
        colorscale=colorscale,
        colorbar_title='Est. Installs',
        visible=(i == 0),
        name=cat
    ))

buttons = []
for i, cat in enumerate(categories):
    exceeds = top5.loc[top5['Category'] == cat, 'Exceeds_1M'].iloc[0]
    label = f"\u2605 {cat} (>1M)" if exceeds else cat
    visibility = [j == i for j in range(len(categories))]
    buttons.append(dict(
        label=label,
        method='update',
        args=[{'visible': visibility},
              {'title': f"Estimated Global Installs — {cat}"}]
    ))

fig.update_layout(
    title=f"Estimated Global Installs — {categories[0]}",
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=0.5, xanchor='center', y=1.15, yanchor='top',
        direction='down'
    )],
    geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
    height=600,
    margin=dict(t=110, b=10, l=10, r=10)
)

fig.show()

## 5. Export for the combined dashboard

In [7]:
geo_df.to_csv('task2_choropleth_data.csv', index=False)
top5.to_csv('task2_top5_categories.csv', index=False)
print('Saved: task2_choropleth_data.csv, task2_top5_categories.csv')

Saved: task2_choropleth_data.csv, task2_top5_categories.csv


### Note on the 6PM–8PM IST display rule
Same as Task 1, this is a dashboard-level display rule (not a data filter), implemented with the same IST time-check logic in the combined `dashboard.html`.